In [1]:
import cv2
import numpy as np
from tqdm import tqdm
from keras.preprocessing.image import load_img, img_to_array
from u_net import get_unet_128
import glob

Using TensorFlow backend.


In [2]:
input_size, model = get_unet_128()

W0826 22:23:26.520982 11704 deprecation_wrapper.py:119] From c:\users\shane\anaconda3\envs\tf\lib\site-packages\keras\backend\tensorflow_backend.py:74: The name tf.get_default_graph is deprecated. Please use tf.compat.v1.get_default_graph instead.

W0826 22:23:26.542981 11704 deprecation_wrapper.py:119] From c:\users\shane\anaconda3\envs\tf\lib\site-packages\keras\backend\tensorflow_backend.py:517: The name tf.placeholder is deprecated. Please use tf.compat.v1.placeholder instead.

W0826 22:23:26.546996 11704 deprecation_wrapper.py:119] From c:\users\shane\anaconda3\envs\tf\lib\site-packages\keras\backend\tensorflow_backend.py:4138: The name tf.random_uniform is deprecated. Please use tf.random.uniform instead.

W0826 22:23:26.572990 11704 deprecation_wrapper.py:119] From c:\users\shane\anaconda3\envs\tf\lib\site-packages\keras\backend\tensorflow_backend.py:174: The name tf.get_default_session is deprecated. Please use tf.compat.v1.get_default_session instead.

W0826 22:23:26.573995 11

In [3]:
model.load_weights(filepath='weights/best_weights.hdf5')

In [4]:
img_path = 'input/test'
img_ext = '.jpg'
img_form = img_path + '/{}' + img_ext

In [5]:
img_names = glob.glob(img_path + '/*' + img_ext)
img_names = [names.replace('\\', '/') for names in img_names]
img_names = [names.split('/')[-1].split('.')[0] for names in img_names]
print('Found {} images.'.format(len(img_names)))

Found 19 images.


In [6]:
for start in tqdm(range(0, len(img_names))):
    x_batch = []
    id = img_names[start]
    img_path = img_form.format(id)
    img = load_img(img_path, target_size = (input_size, input_size))
    img = img_to_array(img)
    x_batch.append(img)
    x_batch = np.array(x_batch, np.float32) / 255
    preds = model.predict_on_batch(x_batch)
    preds = np.squeeze(preds, axis=3)
    for index, pred in enumerate(preds):
        prob = np.array(cv2.resize(pred, (240, 320)) > 0.5).astype(np.float32) * 255
        current_filename = id[index]
        cv2.imwrite('input/test/segmentation/{}.png'.format(id), prob)

100%|██████████████████████████████████████████████████████████████████████████████████| 19/19 [00:03<00:00,  6.75it/s]
